# LSLOD Cloud Analysis

Analysis of the Life Sciences LOD cloud: cross-dataset patterns and shared vocabularies.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

OUTPUT_DIR = Path("/home/javier.millanacosta/rdfsolve/output")
LSLOD_DIR = OUTPUT_DIR / "lslod_cloud"
print(f"LSLOD directory: {LSLOD_DIR}")

In [ ]:
# Load LSLOD cloud schema
lslod_schema_path = LSLOD_DIR / "lslod_cloud_schema.jsonld"
if lslod_schema_path.exists():
    lslod_schema = json.loads(lslod_schema_path.read_text())
    print(f"LSLOD schema loaded: {len(lslod_schema.get('@graph', []))} entries")
else:
    print("LSLOD schema not found - run slurm_lslod_cloud.sh first")
    lslod_schema = None

In [ ]:
if lslod_schema:
    # Analyze patterns in LSLOD cloud
    class_usage = Counter()
    property_usage = Counter()
    cross_class_patterns = []
    
    for item in lslod_schema.get("@graph", []):
        subj_class = item.get("@id")
        if subj_class:
            class_usage[subj_class] += 1
        
        for p in item.get("patterns", []):
            prop = p.get("property")
            obj_class = p.get("object_class")
            
            if prop:
                property_usage[prop] += 1
            
            if subj_class and obj_class and subj_class != obj_class:
                cross_class_patterns.append({
                    "subject_class": subj_class,
                    "property": prop,
                    "object_class": obj_class,
                })
    
    print(f"Unique classes: {len(class_usage)}")
    print(f"Unique properties: {len(property_usage)}")
    print(f"Cross-class patterns: {len(cross_class_patterns)}")

In [ ]:
if lslod_schema:
    print("Top 20 classes by usage:")
    for cls, count in class_usage.most_common(20):
        print(f"  {count:5d}  {cls[:70]}")

In [ ]:
if lslod_schema:
    print("Top 20 properties by usage:")
    for prop, count in property_usage.most_common(20):
        print(f"  {count:5d}  {prop[:70]}")

In [ ]:
if lslod_schema and cross_class_patterns:
    df_patterns = pd.DataFrame(cross_class_patterns)
    print(f"Cross-class patterns: {len(df_patterns)}")
    df_patterns.head(20)

In [ ]:
# Load individual VoID files to see dataset metadata
void_files = list(OUTPUT_DIR.glob("**/*_void.ttl"))
print(f"Found {len(void_files)} VoID files")
for f in void_files[:10]:
    print(f"  - {f.parent.name}/{f.name}")